# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 11: GNN Failure Modes: Oversmoothing & Oversquashing

### 1. Conceptual Intuition & Theoretical Background
While deep CNNs and Transformers thrive with hundreds of layers, standard GNNs (GCN, GAT) degrade significantly when depth exceeds $L > 4$ layers due to two distinct mathematical phenomena:

**1. Oversmoothing (Dirichlet Energy Collapse):**
Repeated Laplacian smoothing causes node representations to converge toward a uniform stationary consensus:
$$h_i^{(l)} \longrightarrow h_j^{(l)}, \quad \forall i, j$$
Measured via **Dirichlet Energy**:
$$\mathcal{E}(H) = \frac{1}{2} \text{Tr}(H^T L_{\text{sym}} H) \longrightarrow 0 \quad \text{as } L \to \infty$$
Node embeddings lose all discriminative power and become indistinguishable.

**2. Oversquashing (Information Bottleneck):**
In graphs with exponential volume growth (tree-like structures or negative Ricci curvature), the size of a node's $k$-hop receptive field grows exponentially:
$$|\mathcal{N}_k(v)| \approx d^k$$
Compressing information from thousands of neighbor nodes into a fixed-size embedding vector creates an information bottleneck that causes gradients from distant nodes to vanish exponentially.

---

### 2. Scratch Python Implementation: Dirichlet Energy Decay Tracker

In [ ]:
import numpy as np

def compute_dirichlet_energy(H, L_sym):
    """Computes Dirichlet Energy: E(H) = 0.5 * Tr(H^T @ L_sym @ H)"""
    return 0.5 * np.trace(np.dot(H.T, np.dot(L_sym, H)))

# Build a Simple Connected Graph (N=10 nodes)
n_nodes = 10
np.random.seed(42)
# Adjacency matrix for a ring with shortcuts
A = np.zeros((n_nodes, n_nodes))
for i in range(n_nodes):
    A[i, (i + 1) % n_nodes] = 1
    A[(i + 1) % n_nodes, i] = 1
A[0, 5] = A[5, 0] = 1 # Shortcut

# Degree and Symmetric Normalized Laplacian
deg = np.sum(A, axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(deg))
L_sym = np.eye(n_nodes) - np.dot(D_inv_sqrt, np.dot(A, D_inv_sqrt))

# Initial random distinct node features
H = np.random.normal(0, 1.0, (n_nodes, 4))

# Track Dirichlet energy over 15 GCN propagation steps
energies = [compute_dirichlet_energy(H, L_sym)]
H_curr = H.copy()

# GCN Normalized adjacency: A_hat = D^(-1/2) A D^(-1/2)
A_hat = np.dot(D_inv_sqrt, np.dot(A, D_inv_sqrt))

for layer in range(15):
    # Pure propagation step without residual connections
    H_curr = np.dot(A_hat, H_curr)
    energies.append(compute_dirichlet_energy(H_curr, L_sym))

print("--- GNN Oversmoothing: Dirichlet Energy Collapse ---")
print(f"Layer 0 Initial Dirichlet Energy:  {energies[0]:.4f}")
print(f"Layer 5 Dirichlet Energy:          {energies[5]:.4f}")
print(f"Layer 15 Dirichlet Energy:         {energies[15]:.6e} (Collapsed near 0!)")
print(f"Energy Reduction Ratio:            {(1 - energies[-1] / energies[0]) * 100:.2f}%")
print("Node representations have smoothed into identical consensus vectors!")

### 3. Step-by-Step Code Walkthrough
1. **Dirichlet Energy Formulation:** Measures total squared feature difference between adjacent connected nodes.
2. **Exponential Smoothing:** Repeated multiplication by $A_{\text{norm}}$ acts as a low-pass filter, flattening high-frequency feature differences.
3. **Mitigations:** Residual connections, jumping knowledge networks, and DropEdge help counter oversmoothing.

---

### 4. Industry Applications & Trade-offs
- **Protein Structure & Molecular GNNs:** Mitigating oversquashing using graph rewiring (Ricci curvature flow) or virtual supernodes.
- **Trade-off:** Adding virtual global nodes alleviates oversquashing but increases oversmoothing risk.

---

### 5. Top Interview Questions & Answers
**Q1: What is the fundamental difference between Oversmoothing and Oversquashing?**
> **Answer:** Oversmoothing occurs when deep propagation washes out node distinctions, collapsing all embeddings to the stationary mean. Oversquashing occurs when an exponentially large receptive field is compressed into a fixed-size vector, preventing long-range signal propagation.

**Q2: How does the DropEdge technique mitigate oversmoothing?**
> **Answer:** DropEdge randomly removes a fraction of graph edges during each training epoch. This acts as a message-passing regularizer and prevents the Graph Laplacian from collapsing features into a uniform subspace.